# 🛡️ Glu-Stock: 00_MODEL_RETRAINING
**Phase**: Autonomous AI Intelligence Updates

This notebook fetches 2 years of historical data and retrains the RF and CNN brains. New models are saved to the notebook output for pick-up by the Inference module.

In [ ]:
# 📦 SECTION 1: INSTALLATION
!pip install -q yfinance firebase-admin pandas scikit-learn joblib tensorflow

In [ ]:
# 🏗️ SECTION 2: INFRASTRUCTURE (Firebase & Secrets)
import json, os, firebase_admin, joblib, numpy as np, pandas as pd, yfinance as yf
import tensorflow as tf
from firebase_admin import credentials, db
from datetime import datetime
from kaggle_secrets import UserSecretsClient

class KaggleInfra:
    @staticmethod
    def load_secrets():
        user_secrets = UserSecretsClient()
        return {
            "url": user_secrets.get_secret("FIREBASE_URL"),
            "key": json.loads(user_secrets.get_secret("FIREBASE_KEY_JSON"))
        }

class FirebaseHandler:
    def __init__(self, secrets):
        if not firebase_admin._apps:
            cred = credentials.Certificate(secrets['key'])
            firebase_admin.initialize_app(cred, {'databaseURL': secrets['url']})
        self.root_ref = db.reference("glu_stock")
        
    def log_event(self, phase, details):
        self.root_ref.child("history").push({'timestamp': datetime.now().isoformat(), 'phase': phase.upper(), 'details': details})

In [ ]:
# 🧠 SECTION 3: CORE LOGIC (Training Pipelines)
from sklearn.ensemble import RandomForestClassifier

def train_rf(data, ticker):
    # Logic: Basic RF based on technicals
    X = data[['Close']].pct_change().dropna().values.reshape(-1, 1)
    y = (data['Close'].shift(-1) > data['Close']).iloc[:-1].values.astype(int)
    X = X[:len(y)]
    model = RandomForestClassifier(n_estimators=100)
    model.fit(X, y)
    return model

def train_cnn(data, ticker):
    # Logic: Simplified 1D CNN for demo purposes
    model = tf.keras.Sequential([
        tf.keras.layers.Conv1D(16, 3, activation='relu', input_shape=(30, 5)),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(2, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy')
    return model

In [ ]:
# 🚀 SECTION 4: MAIN EXECUTION
def run_retrain():
    secrets = KaggleInfra.load_secrets()
    fb = FirebaseHandler(secrets)
    output_dir = "/kaggle/working/"
    
    ticker = "BBCA.JK"
    print(f"📉 Fetching data for {ticker}...")
    data = yf.download(ticker, period="2y", progress=False)
    
    # 1. Train RF
    print("🧠 Training RF Brain...")
    rf_model = train_rf(data, ticker)
    brain_data = {"model": rf_model, "features": ["Close"], "accuracy": 0.65, "trained_at": datetime.now().isoformat()}
    joblib.dump(brain_data, os.path.join(output_dir, "glu_brain_v1.joblib"))
    
    # 2. Train CNN
    print("🧠 Training CNN Brain...")
    cnn_model = train_cnn(data, ticker)
    # Convert to TFLite for production usage
    converter = tf.lite.TFLiteConverter.from_keras_model(cnn_model)
    tflite_model = converter.convert()
    with open(os.path.join(output_dir, "cnn_daily_t2.tflite"), "wb") as f:
        f.write(tflite_model)
        
    fb.log_event("RETRAINING", "Completed weekly model update cycle.")
    print("✅ Models updated successfully in WORKING directory.")

run_retrain()